# Titanic: Machine Learning from Disaster
**Author:** Koketso Raphasha | [Kaggle: Raphasha27](https://kaggle.com/Raphasha27)

**Best Public Score:** 77.5% (v4)

**Approach:** KNN imputation, interaction features (Age×Pclass, Fare×Pclass), LogisticRegressionCV with L1/L2 regularization

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')
print(f'Train: {train.shape}, Test: {test.shape}')

## Feature Engineering

In [ ]:
def make_features(df):
    data = df.copy()
    data['Title'] = data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    data['Title'] = data['Title'].map({
        'Mr':'Mr','Miss':'Miss','Mrs':'Mrs','Master':'Master'
    }).fillna('Rare')
    title_map = {'Mr':0,'Miss':1,'Mrs':2,'Master':3,'Rare':4}
    data['Title'] = data['Title'].map(title_map).fillna(4).astype(int)
    
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    data['HasCabin'] = data['Cabin'].notna().astype(int)
    data['Sex'] = (data['Sex'] == 'male').astype(int)
    data['Embarked'] = data['Embarked'].fillna('S').map({'S':0,'C':1,'Q':2}).fillna(0).astype(int)
    data['Pclass'] = data['Pclass'].astype(int)
    data['FamilyCat'] = pd.cut(data['FamilySize'], bins=[0,1,4,20], labels=[0,1,2]).astype(int)
    
    return data

In [ ]:
train_feat = make_features(train)
test_feat = make_features(test)

## KNN Imputation for Age

In [ ]:
full = pd.concat([train_feat, test_feat], ignore_index=True)
age_cols = ['Age','Pclass','Sex','Fare','SibSp','Parch','Title','FamilySize']
age_data = full[age_cols].copy()

knn = KNNImputer(n_neighbors=5)
full['Age'] = pd.DataFrame(knn.fit_transform(age_data), columns=age_cols)['Age']

fare_med = full.groupby('Pclass')['Fare'].transform('median')
full['Fare'] = full['Fare'].fillna(fare_med)

## Interaction Features

In [ ]:
full['Age_Pclass'] = full['Age'] * full['Pclass']
full['Fare_Pclass'] = full['Fare'] / (full['Pclass'] + 1)

## Prepare Training Data

In [ ]:
feature_cols = ['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked','Title',
                'FamilySize','IsAlone','HasCabin','Age_Pclass','Fare_Pclass','FamilyCat']

X_full = full.iloc[:len(train)][feature_cols]
X_test = full.iloc[len(train):][feature_cols]
y_full = train['Survived'].values

scaler = StandardScaler()
X_full_s = scaler.fit_transform(X_full)
X_test_s = scaler.transform(X_test)

print(f'Features: {len(feature_cols)}')

## Train Model

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

model = LogisticRegressionCV(
    Cs=[0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0],
    cv=5, penalty='l2', solver='liblinear',
    max_iter=3000, random_state=42
)

scores = cross_val_score(model, X_full_s, y_full, cv=cv, scoring='accuracy')
model.fit(X_full_s, y_full)
print(f'CV Accuracy: {scores.mean():.4f} (+/- {scores.std()*2:.4f})')

## Generate Submission

In [ ]:
preds = model.predict(X_test_s).astype(int)

sub = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': preds
})
sub.to_csv('submission.csv', index=False)
print(f'Survived: {preds.sum()}/{len(preds)} ({preds.mean()*100:.1f}%)')
print('Ready: submission.csv')